# Feature Engineering and Preprocessing Pipeline
This notebook implements the complete feature engineering, data transformation (scaling and encoding), train-test splitting, and class imbalance resampling (using SMOTE) for both the Fraud and Credit Card datasets.

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.model_selection import train_test_split

# Add src directory to path
sys.path.append(os.path.abspath('../'))
from src.data_preprocessor import load_data, clean_fraud_data, clean_creditcard_data, merge_geolocation
from src.feature_engineering import (
    extract_time_features,
    calculate_velocity_features,
    preprocess_countries,
    one_hot_encode,
    scale_features
)
from src.sampling import resample_data

os.makedirs('../data/processed', exist_ok=True)

## 1. Load and Clean Raw Datasets

In [2]:
# Load Fraud data
fraud_raw = load_data('../data/raw/Fraud_Data.csv')
ip_raw = load_data('../data/raw/IpAddress_to_Country.csv')

# Clean and merge geolocation
fraud_cleaned = clean_fraud_data(fraud_raw)
fraud = merge_geolocation(fraud_cleaned, ip_raw)

# Load Credit Card data
credit_raw = load_data('../data/raw/creditcard.csv')
credit = clean_creditcard_data(credit_raw)

print('Fraud shape:', fraud.shape)
print('Credit shape:', credit.shape)

Removed 1081 duplicate rows from Credit Card data.
Fraud shape: (151112, 12)
Credit shape: (283726, 31)


## 2. Feature Engineering (Fraud_Data.csv)
We extract time features (`hour_of_day`, `day_of_week`, `time_since_signup`) and calculate transaction velocity features (`device_sharing_count`, `ip_sharing_count`, rolling 1h/24h counts).

In [3]:
print('Engineering features for Fraud dataset...')
fraud_featured = extract_time_features(fraud)
fraud_featured = calculate_velocity_features(fraud_featured)
print('Finished feature engineering. Columns added:', [col for col in fraud_featured.columns if col not in fraud.columns])
fraud_featured.head()

Engineering features for Fraud dataset...


Finished feature engineering. Columns added: ['hour_of_day', 'day_of_week', 'time_since_signup', 'device_tx_count_1h', 'device_tx_count_24h', 'ip_tx_count_1h', 'ip_tx_count_24h', 'device_sharing_count', 'ip_sharing_count']


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,...,country,hour_of_day,day_of_week,time_since_signup,device_tx_count_1h,device_tx_count_24h,ip_tx_count_1h,ip_tx_count_24h,device_sharing_count,ip_sharing_count
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,...,Japan,2,5,4506682.0,1.0,1.0,1.0,1.0,1,1
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,...,United States,1,0,17944.0,1.0,1.0,1.0,1.0,1,1
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,...,United States,18,3,1.0,11.0,11.0,11.0,11.0,12,12
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,...,Unknown,13,0,492085.0,1.0,1.0,1.0,1.0,1,1
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,...,United States,18,2,4361461.0,1.0,1.0,1.0,1.0,1,1


## 3. Train-Test Splits
We separate features and targets and run an 80/20 train/test split. Crucially, all encoding, scaling, and resampling fits will occur ONLY on the training split to prevent leakage.

In [4]:
# Fraud split
X_fraud = fraud_featured.drop(columns=['user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address', 'class'])
y_fraud = fraud_featured['class']

X_fraud_train, X_fraud_test, y_fraud_train, y_fraud_test = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

# Credit split
X_credit = credit.drop(columns=['Class'])
y_credit = credit['Class']

X_credit_train, X_credit_test, y_credit_train, y_credit_test = train_test_split(
    X_credit, y_credit, test_size=0.2, random_state=42, stratify=y_credit
)

print('Fraud train shape:', X_fraud_train.shape, 'test shape:', X_fraud_test.shape)
print('Credit train shape:', X_credit_train.shape, 'test shape:', X_credit_test.shape)

Fraud train shape: (120889, 15) test shape: (30223, 15)
Credit train shape: (226980, 30) test shape: (56746, 30)


## 4. Categorical Encoding & Scaling
We preprocess countries (retaining the top 15 and grouping others into 'Other'), one-hot encode categoricals, and apply standard scaling to all numerical features.

In [5]:
# Categorical preprocessing for Fraud dataset country column
X_fraud_train, X_fraud_test = preprocess_countries(X_fraud_train, X_fraud_test, top_n=15)

# One-hot encode categoricals: source, browser, sex, country
categorical_cols = ['source', 'browser', 'sex', 'country']
X_fraud_train_enc, X_fraud_test_enc = one_hot_encode(X_fraud_train, X_fraud_test, categorical_cols)

# Standardize numerical features for Fraud
numerical_cols_fraud = [
    'purchase_value', 'age', 'hour_of_day', 'day_of_week', 'time_since_signup',
    'device_tx_count_1h', 'device_tx_count_24h', 'ip_tx_count_1h', 'ip_tx_count_24h',
    'device_sharing_count', 'ip_sharing_count'
]
X_fraud_train_scaled, X_fraud_test_scaled, fraud_scaler = scale_features(
    X_fraud_train_enc, X_fraud_test_enc, numerical_cols_fraud, 'standard'
)

# Standardize numerical features for Credit (all features are numerical)
numerical_cols_credit = X_credit_train.columns.tolist()
X_credit_train_scaled, X_credit_test_scaled, credit_scaler = scale_features(
    X_credit_train, X_credit_test, numerical_cols_credit, 'standard'
)

print('Fraud features shape after scaling/encoding:', X_fraud_train_scaled.shape)
print('Credit features shape after scaling:', X_credit_train_scaled.shape)

Fraud features shape after scaling/encoding: (120889, 33)
Credit features shape after scaling: (226980, 30)


## 5. Handle Class Imbalance
We apply SMOTE (Synthetic Minority Over-sampling Technique) to the training sets only, documenting the class distribution shift.

In [6]:
print('Resampling Fraud Training Set...')
X_fraud_train_res, y_fraud_train_res = resample_data(X_fraud_train_scaled, y_fraud_train, method='smote')

print('\nResampling Credit Card Training Set...')
# Note: For Credit Card, SMOTE creates ~226k synthetic positive instances to balance classes.
X_credit_train_res, y_credit_train_res = resample_data(X_credit_train_scaled, y_credit_train, method='smote')

print('\nResampling complete!')

Resampling Fraud Training Set...
Class distribution before resampling:
class
0    0.906352
1    0.093648
Name: proportion, dtype: float64
class
0    109568
1     11321
Name: count, dtype: int64


Class distribution after resampling (smote):


class
0    0.5
1    0.5
Name: proportion, dtype: float64
class
0    109568
1    109568
Name: count, dtype: int64

Resampling Credit Card Training Set...
Class distribution before resampling:
Class
0    0.998335
1    0.001665
Name: proportion, dtype: float64
Class
0    226602
1       378
Name: count, dtype: int64


Class distribution after resampling (smote):
Class
0    0.5
1    0.5
Name: proportion, dtype: float64
Class
0    226602
1    226602
Name: count, dtype: int64

Resampling complete!


## 6. Save Processed Datasets
We export the final train/test datasets to the `data/processed/` directory.

In [7]:
# Save Fraud processed splits
X_fraud_train_res.to_csv('../data/processed/fraud_X_train_resampled.csv', index=False)
y_fraud_train_res.to_csv('../data/processed/fraud_y_train_resampled.csv', index=False)
X_fraud_test_scaled.to_csv('../data/processed/fraud_X_test.csv', index=False)
y_fraud_test.to_csv('../data/processed/fraud_y_test.csv', index=False)

# Save Credit processed splits
X_credit_train_res.to_csv('../data/processed/credit_X_train_resampled.csv', index=False)
y_credit_train_res.to_csv('../data/processed/credit_y_train_resampled.csv', index=False)
X_credit_test_scaled.to_csv('../data/processed/credit_X_test.csv', index=False)
y_credit_test.to_csv('../data/processed/credit_y_test.csv', index=False)

print('Saved preprocessed splits successfully!')
print('fraud_X_train_resampled shape:', X_fraud_train_res.shape)
print('credit_X_train_resampled shape:', X_credit_train_res.shape)

Saved preprocessed splits successfully!
fraud_X_train_resampled shape: (219136, 33)
credit_X_train_resampled shape: (453204, 30)
